# B1 document hierarchy

**Later illustrated companion; not the executed training notebook.** Historical metrics below are quoted from copied run records, never represented as notebook outputs.

A budget-paused run is not converged; the target was four epochs with checkpoint selection and train-only calibration. The v7 source also defines gated depth expansion and an upper FFN option, but the recorded configs for these exports keep encoder_depth=6 and expand_ffn=false; those unlaunched options have no reported score. Original checkpoint bytes were downloaded and verified in the local archive, but are omitted from GitHub.

![Preserved computation graph](../figures/architecture.svg)

## Task and architecture

B1 6-encoder/6-decoder original Phase 1 backbone; 512 tokens per chunk, 16 ordered chunks per case, overlap 100, tail sampling. Two document Transformer blocks followed by bidirectional GRU. Token representation uses a learned mixture of mean, attention and last content token; position enters before document aggregation. The final learned attention pools document states for two-class logits.

**Adaptation:** Staged B1 fine-tuning; document head mode `transformer_gru`

Data path: official train → training-only fit/selection → official labeled dev report → 1,517-label-free test predictions. The original `code/` and `evidence/` directories contain the audit trail.

## Exact mechanism and experimental controls

### Provenance and scientific boundary

[CODE] B1: six-layer encoder and six-layer decoder from own Phase 1 weights; decoder representations supply token evidence.

[CODE] After token evidence pooling, a width-256 projection plus normalized position features enters two document Transformer blocks (four heads, width-768 gated feed-forward inner layer), then a 128-unit-per-direction bidirectional GRU. A learned document attention pools states into two logits.

[CODE/RUN] Staged fine-tuning starts at the task head and progressively unfreezes original encoder/decoder blocks. All optimizer parameter groups exist from the beginning; stage changes control which tensors receive gradients. Saved optimizer, scaler, RNG and position permit continuation between segments without reinitializing this head.

### Exact recorded configuration

| Setting | Recorded value |
|---|---|
| Backbone | `B1: six-layer encoder and six-layer decoder from own Phase 1 weights; decoder representations supply token evidence.` |
| Head mode | `transformer_gru` |
| Window policy | `tail` |
| Chunking | `16 × 512 tokens, overlap 100` |
| Document width | `256 for non-reference heads; B1 width for reference` |
| Learning rates | `backbone 1e-05; head 0.0003` |
| Accumulation | `8` |
| Warmup updates | `100` |
| Target epochs | `4` |
| Completed epochs | `2` |
| Optimizer steps | `8000` |
| Selected checkpoint | `0626c85191eb9e74a2133152e336b2b0cf9ad3e9a082b315eb2d129f3ac10a90` |

### Optimization and selection

[CODE] The B1 study uses cross-entropy for the binary label, AdamW with 0.01 weight decay, and staged unfreezing. Validation is performed on a fixed train-only holdout; checkpoint selection requires at least two epochs before patience can declare convergence. A time-budget stop instead records paused and a resumable checkpoint.

### Evaluation protocol

[RUN] The fixed 512-example train holdout selected model checkpoints at threshold 0.5. A separate, group-disjoint 512-example train calibration set selected the final decision threshold. The official 994-case dev evaluation used that threshold: macro-F1 0.534473; using 0.5 instead gave 0.499201. The split code excludes normalized-text duplicate groups across train, holdout, and calibration. The 1,517 test cases supplied no labels. A budget-paused run is not convergence.

### Reproduction lineage

B1 source and notebook versions are archived. The v7 source snapshot in code/source/ is the latest executable reference; for older v4 exports, the corresponding versioned notebook and run config identify the exact earlier run rather than implying that v7 code executed v4.

## Recorded result

Version 4 run state **complete**; 8000 optimizer updates and 2 complete epochs, selected step 6500. Calibrated official 994-case dev macro-F1 **0.534473** at threshold 0.486789; default 0.5 dev macro-F1 0.499201. Selected checkpoint SHA-256 `0626c85191eb9e74a2133152e336b2b0cf9ad3e9a082b315eb2d129f3ac10a90`; verified 1,517-ID ZIP SHA-256 `eb70fc73838a3fa84a6bd6ef5eac03ce680f62ef93bf780e8a9775ea7a384d7a`.

Local official dev, calibration CV, and Codabench scores are different measurements. A historical score is not an output of this notebook.

## Preserved run trace

![Recorded training loss or fitted stack coefficients](../figures/recorded_trace.svg)

The figure is drawn from copied run records, not from an execution of this companion notebook. The machine-readable source is in `evidence/`.

## Source excerpt: `model.py`

The complete source is in `code/`. This excerpt illustrates the preserved implementation; it is not a training replay.

In [ ]:
import copy
from dataclasses import replace

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

from phase3_rr.models import load_backbone
from phase4_cjpe.models import EvidenceCaseModel


class GatedEncoderBlock(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.block = copy.deepcopy(block)
        self.attention_gate = nn.Parameter(torch.zeros(()))
        self.ffn_gate = nn.Parameter(torch.zeros(()))

    def forward(self, x, valid_mask):
        x = x + self.attention_gate * self.block.attn(self.block.attn_norm(x), valid_mask)
        return x + self.ffn_gate * self.block.ffn(self.block.ffn_norm(x))


def grow_encoder(backbone):
    if len(backbone.encoder) != 6:
        raise ValueError('Depth expansion requires the original six encoder blocks')
    layers, mapping = [], []
    for i, block in enumerate(backbone.encoder):
        layers.append(block)
        mapping.append({'new_index': len(layers) - 1, 'original_index': i, 'gated_copy': False})
        if i in (2, 5):
            layers.append(GatedEncoderBlock(block))
            mapping.append({'new_index': len(layers) - 1, 'original_index': i, 'gated_copy': True})
    backbone.encoder = nn.ModuleList(layers)
    backbone.cfg = replace(backbone.cfg, n_encoder_layers=8)
    return mapping


def expand_upper_ffn(backbone, new_inner=3072):
    names = []
    targets = [('encoder', i, b) for i, b in enumerate(backbone.encoder)][-2:]
    targets += [('decoder', i, b) for i, b in enumerate(backbone.decoder)][-2:]
    for family, index, block in targets:
        block = block.block if isinstance(block, GatedEncoderBlock) else block
        ffn = block.ffn
        old, width = ffn.w1.weight.shape
        if new_inner <= old:
            raise ValueError('FFN expansion must increase inner width')
        device, dtype = ffn.w1.weight.device, ffn.w1.weight.dtype
        first = nn.Linear(width, new_inner, bias=False, device=device, dtype=dtype)
        third = nn.Linear(width, new_inner, bias=False, device=device, dtype=dtype)
        second = nn.Linear(new_inner, width, bias=False, device=device, dtype=dtype)
        with torch.no_grad():
            first.weight[:old].copy_(ffn.w1.weight)
            third.weight[:old].copy_(ffn.w3.weight)
            second.weight.zero_()
            second.weight[:, :old].copy_(ffn.w2.weight)
        ffn.w1, ffn.w2, ffn.w3 = first, second, third
        names.append(f'{family}.{index}.ffn:{old}->{new_inner}')
    return names


class DocumentBlock(nn.Module):
    def __init__(self, width=256, heads=4, inner=768, dropout=.1):
        super().__init__()
        self.heads, self.dropout = heads, dropout
        self.attention_norm, self.ffn_norm = nn.LayerNorm(width), nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.output = nn.Linear(width, width)
        self.up, self.gate = nn.Linear(width, inner), nn.Linear(width, inner)
        self.down = nn.Linear(inner, width)

    def forward(self, x):
        b, length, width = x.shape
        q, k, v = [a.reshape(b, length, self.heads, width // self.heads).transpose(1, 2)
                   for a in self.qkv(self.attention_norm(x)).chunk(3, -1)]
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.)
        x = x + F.dropout(self.output(out.transpose(1, 2).reshape(b, length, width)), self.dropout, self.training)
        z = self.ffn_norm(x)
        return x + F.dropout(self.down(F.silu(self.gate(z)) * self.up(z)), self.dropout, self.training)


class B1Study(EvidenceCaseModel):
    def __init__(self, backbone, config):
        width = backbone.cfg.d_model
        super().__init__(backbone, 'b1', width)
        self.study_config = config
        self.mode = config['head_mode']
        self.chunk_batch = config['chunk_batch']
        self.depth_mapping = grow_encoder(backbone) if config.get('encoder_depth', 6) == 8 else []
        self.ffn_mapping = expand_upper_ffn(backbone) if config.get('expand_ffn', False) else []
        if self.mode != 'reference':
            for name in ('pool', 'head', 'position', 'chunk_context', 'context_projection',
                         'context_scale', 'context_norm', 'document_mix'):
                delattr(self, name)
            self.projection = nn.Linear(width, 256)
            self.document_position = nn.Linear(2, 256, bias=False)
            self.document_blocks = nn.ModuleList([DocumentBlock() for _ in range(2)]) if self.mode != 'gru_only' else nn.ModuleList()
            self.document_gru = nn.GRU(256, 128, bidirectional=True, batch_first=True) if self.mode != 'transformer_only' else None
            self.document_norm = nn.LayerNorm(256)
            self.document_attention = nn.Sequential(nn.Linear(256, 64), nn.Tanh(), nn.Linear(64, 1))
            self.classifier = nn.Sequential(nn.LayerNorm(256), nn.Dropout(.15), nn.Linear(256, 2))

    def feature_batch(self, ids, valid):
        m = self.backbone
        memory, valid = m.encode(ids, valid)
        x = m.tok_embeddings(ids)
        # Right padding cannot influence valid decoder positions through causal self-attention.
        for block in m.decoder:

## Verify the copied run evidence

The next cell is a read-only inspection helper. It does not train or generate new scores. Running it is optional; this delivered notebook has no saved execution output.

In [ ]:
from pathlib import Path
import json
root = Path('..')
for record in sorted((root / 'evidence').glob('*terminal.json')):
    result = json.loads(record.read_text(encoding='utf-8'))
    print(record.name, result.get('name'), result.get('state'), result.get('dev_metrics', {}).get('macro_f1'))


## Reproduction boundary

The package has code, configs, selected records, and prediction ZIP. Original model weights and official data are retained in the local archives, not duplicated here. Paths in the original configs must be remapped before an authorized rerun. The source code can be reviewed without executing it.